# Seleção por série e validação repetida da confiança (#12)

Lê a rodada `3051af2281b74935981b7ccf6d98e7f8` no MLflow, sem treinar nada. Só backtest: o OOT segue selado. Critério: `docs/relatorio/fase2/12-pre-registro.md` (D-018). Desvio: `docs/relatorio/fase2/12.md`.

**Status:** conclusões PROPOSTAS até a aprovação de D-019 e D-020.

In [1]:
import os
from collections import Counter
from pathlib import Path

import mlflow
import pandas as pd
from mlflow import MlflowClient

from case_mr_health.config import Settings
from case_mr_health.selecao import carregar_rodada, selecionar, sem_pos_hoc

RODADA = '3051af2281b74935981b7ccf6d98e7f8'
ROOT = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / 'pyproject.toml').is_file()
)
os.chdir(ROOT)  # caminhos da configuração são relativos à raiz
cfg = Settings()  # só a visão de desenvolvimento e o MLflow são abertos
series = carregar_rodada(cfg, RODADA)
cliente = MlflowClient()
print('séries:', len(series), '| runs:', sum(map(len, series.values())))

/home/rwallan/git/case-mr-health/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/rwallan/git/case-mr-health/.venv/lib/python3.14/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


séries: 8 | runs: 72


## 1. Regra pré-registrada e desvio

Pergunta: qual modelo vence em cada série, com e sem o naive nas diárias?

In [2]:
def tabela(filtrar):
    linhas = []
    for (sku, gran), runs in sorted(series.items()):
        s = selecionar(sku, gran, filtrar(gran, runs))
        v, b = s.vencedor, s.baseline
        linhas.append({
            'série': f'{sku} {gran}',
            'vencedor': v.algoritmo,
            'baseline': b.algoritmo,
            'MAE': round(v.mae, 3),
            'MAE baseline': round(b.mae, 3),
            'MASE': round(v.medias['mase'], 3),
            'WAPE': round(v.medias['wape'], 3),
            'viés rel.': round(v.medias.get('vies_relativo', float('nan')), 2),
            'cob. 80': round(v.medias['cobertura_80'], 2),
            'cob. 95': round(v.medias['cobertura_95'], 2),
            'superam': sum(c.supera for c in s.comparacoes),
            'comparados': len(s.comparacoes),
        })
    return pd.DataFrame(linhas)


pre_registrada = tabela(lambda _g, runs: runs)
pre_registrada

,série,vencedor,baseline,MAE,MAE baseline,MASE,WAPE,viés rel.,cob. 80,cob. 95,superam,comparados
0,item A diaria,naive_sazonal,naive_sazonal,2.143,2.143,0.811,0.897,-0.10,1.00,1.00,0,12
1,item A semanal,media_historica,media_historica,3.933,3.933,0.664,0.267,-0.04,0.75,1.00,0,1
2,item B diaria,naive,naive,2.143,2.143,0.744,1.375,-0.31,0.93,0.93,0,12
3,item B semanal,media_historica,media_historica,6.750,6.750,0.605,0.687,0.32,1.00,1.00,0,1
4,item C diaria,naive,naive,2.343,2.343,0.836,0.880,-0.66,0.80,0.97,0,12
5,item C semanal,media_historica,media_historica,9.236,9.236,1.877,0.430,-0.22,0.50,0.75,0,1
6,item D diaria,sarima,media_historica,2.033,2.072,0.682,0.730,-0.04,0.93,0.93,2,12
7,item D semanal,media_historica,media_historica,4.699,4.699,0.963,0.205,-0.02,0.75,1.00,0,1


In [3]:
com_desvio = tabela(sem_pos_hoc)
com_desvio

,série,vencedor,baseline,MAE,MAE baseline,MASE,WAPE,viés rel.,cob. 80,cob. 95,superam,comparados
0,item A diaria,naive_sazonal,naive_sazonal,2.143,2.143,0.811,0.897,-0.10,1.00,1.00,0,12
1,item A semanal,media_historica,media_historica,3.933,3.933,0.664,0.267,-0.04,0.75,1.00,0,1
2,item B diaria,tsb,naive_sazonal,2.369,2.536,0.821,1.592,0.25,0.89,0.93,3,12
3,item B semanal,media_historica,media_historica,6.750,6.750,0.605,0.687,0.32,1.00,1.00,0,1
4,item C diaria,media_historica,media_historica,2.813,2.813,1.005,1.111,-0.26,0.83,1.00,0,12
5,item C semanal,media_historica,media_historica,9.236,9.236,1.877,0.430,-0.22,0.50,0.75,0,1
6,item D diaria,sarima,media_historica,2.033,2.072,0.682,0.730,-0.04,0.93,0.93,2,12
7,item D semanal,media_historica,media_historica,4.699,4.699,0.963,0.205,-0.02,0.75,1.00,0,1


**Observação.** Com a regra pré-registrada, 7 das 8 séries ficam com um baseline. Só o item D diário troca (SARIMA, ganho pareado de 1,9%), e nas semanais nenhum candidato supera a média histórica. Em B e C diárias vence o naive, com viés de -31% e -66%. Com o desvio, B diária vai para TSB (`alfa_beta` 0,2) e C diária para a média histórica; A e D não mudam.

**Interpretação (hipótese).** O naive venceu pelo MAE em séries intermitentes, nas quais a métrica favorece previsões baixas. Com 4 a 5 folds, o ganho do SARIMA em D é pequeno e pode ser acaso entre 12 candidatos. Nas semanais, a média parece bastar, mas 4 pontos dizem pouco.

## 2. Validação repetida da confiança

Pergunta: as faixas discriminam nos 72 runs (D1 a D3 do `9-pre-registro.md`) e nos 8 selecionados (I1 a I3)?

In [4]:
ids = [
    cliente.get_experiment_by_name(f'demanda-{g}').experiment_id
    for g in ('diaria', 'semanal')
]
runs = cliente.search_runs(
    ids, f"tags.rodada = '{RODADA}'", max_results=1000
)
status = Counter(r.data.tags['confianca'] for r in runs)
fora = sum(n for s, n in status.items() if s != 'Insuficiente')
semanais = Counter(
    r.data.tags['confianca']
    for r in runs
    if r.data.tags['granularidade'] == 'semanal'
)
print('72 runs:', dict(status))
print('semanais:', dict(semanais))
print('D1 (2+ faixas):', len(set(status) - {'Insuficiente'}) >= 2)
print('D2 (nenhum status >= 80%):', max(status.values()) / len(runs) < 0.8)
print('D3 (>= 50% fora de Insuficiente):', fora / len(runs) >= 0.5)

72 runs: {'Insuficiente': 45, 'Boa': 5, 'Média': 10, 'Ruim': 12}
semanais: {'Insuficiente': 12}
D1 (2+ faixas): True
D2 (nenhum status >= 80%): True
D3 (>= 50% fora de Insuficiente): False


In [5]:
selecionados = {}
for (sku, gran), lista in sorted(series.items()):
    s = selecionar(sku, gran, sem_pos_hoc(gran, lista))
    selecionados[f'{sku} {gran}'] = s.vencedor.confianca
fora = [v for v in selecionados.values() if v != 'Insuficiente']
print(selecionados)
print('fora de Insuficiente:', len(fora), '| faixas:', set(fora))
print('I1:', len(fora) < 4)
print('I2:', len(set(fora)) < 2)
print('I3:', not any(
    v != 'Insuficiente' for k, v in selecionados.items() if 'semanal' in k
))

{'item A diaria': 'Média', 'item A semanal': 'Insuficiente', 'item B diaria': 'Insuficiente', 'item B semanal': 'Insuficiente', 'item C diaria': 'Ruim', 'item C semanal': 'Insuficiente', 'item D diaria': 'Insuficiente', 'item D semanal': 'Insuficiente'}
fora de Insuficiente: 2 | faixas: {'Ruim', 'Média'}
I1: True
I2: False
I3: True


**Observação.** Nos 72 runs, D1 e D2 passam e D3 falha, com 45 runs Insuficiente (62,5%), incluindo os 12 semanais. Nos 8 selecionados, só 2 ficam fora de Insuficiente e nenhum é semanal: I1 e I3 disparam, e a confiança é inútil pelo critério pré-registrado.

**Interpretação (hipótese).** O limite é o número de folds, não a qualidade dos candidatos: incluir candidatos não mudou o quadro dos baselines (`fase2/9.md`). A recomendação do ADR 0004 usa a série semanal, justamente onde a confiança nunca classifica.

## 3. Limitações

Backtest com 4 a 5 folds, sem IC de superioridade; régua de 1 ep é freio, não prova. A seleção com desvio foi decidida depois de ver os resultados, antes do OOT. MAE favorece previsões baixas em contagens. Cobertura semanal com 4 pontos (item C: 0,50 e 0,75). Nada aqui usa o OOT.